# Crivo: correções, hipóteses e confirmações

Esta rodada usa somente o transformer, os pesos e o tokenizer próprios do Crivo. O notebook carrega candidatos já treinados; não exige que você treine de novo nem use GPU. Os resultados e as falhas estão conservados no relatório abaixo. São testes sintéticos de preço e requisitos, com interpretação neural, cópia de fontes e cálculo determinístico. Isso não certifica conversa livre.

Os dois candidatos do treino primário falharam nos critérios. Há também um piloto posterior que combina a escolha de fontes do candidato eventos com o escopo dos pesos anteriores. Ele usa duas redes próprias e ainda não está aprovado para o chat.


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/CRIVO-eventos')
BRANCH='codex/correcoes-hipoteses-20261008'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/HROSONE/CRIVO.git',str(REPO)])
else:
    b=subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
    if b!=BRANCH:raise RuntimeError('Pasta de outra branch; escolha uma pasta nova.')
print('Revisão:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
LAB=REPO/'experimentos/escopo_eventos_20261008'


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.2,<3','numpy>=1.24,<3','tokenizers>=0.20,<1'])


## Resultados completos

O modelo anterior é medido com a preparação antiga e nova. Os dois novos candidatos recebem os mesmos lotes; o candidato eventos aumenta conjuntamente a supervisão de escopo e de eventos. A escolha dos pesos usa apenas validação. O teste final não escolhe nem ajusta pesos. Nenhum candidato é ativado automaticamente no chat.


In [ ]:
r=json.loads((LAB/'avaliacao/resumo.json').read_text())
for nome,paineis in r['resultados'].items():
    m=paineis['novo_limitado']
    print(nome,'contratos:',m['contratos_corretos'],'/',m['n'],
          'sessões completas:',m['sessoes_completas'],'/',m['sessoes'])
    print('Eventos:',{k:round(v['acuracia_contrato']*100,1) for k,v in m['por_evento'].items()})
print('Critérios:',json.dumps(r['criterios'],ensure_ascii=False,indent=2))
print('Limites:',r['limites'])


## Piloto posterior: preservar escopo

A regra de composição e um novo painel de 222 casos foram registrados antes das previsões desse piloto. O diagnóstico do teste primário motivou essa alternativa. Não houve outro treino; são os mesmos pesos próprios em duas redes. Frases e entidades são novas, mas autoria, gramática e trajetórias são compartilhadas. Perguntas Compare favorecem o normalizador; a melhora não certifica conversa livre.


In [ ]:
PILOTO=REPO/'experimentos/piloto_escopo_preservado_20261008'
rp=json.loads((PILOTO/'resultados/resumo.json').read_text())
for nome,v in rp['resultados'].items():
    m=v['limitado'];print(nome,m['contratos_corretos'],'/',m['n'], 'sessões:',m['sessoes_completas'],'/',m['sessoes'])
    print({k:(x['contratos_corretos'],x['n']) for k,x in m['por_evento'].items()})
print('Critério do piloto:',rp['criterio_piloto_passou'],'; aprovado para chat:',rp['aprovado_para_chat'])


## Inspecionar uma sequência própria

Edite as falas e execute. A célula mostra os argumentos copiados, a fonte no histórico e se o modelo está tratando a situação como hipótese. A cabeça de eventos só fornece uma previsão auxiliar; não substitui o escopo pelo alvo correto. Fontes ou cálculos executáveis podem estar semanticamente errados. Não há redação livre nesta célula.

Nomes de uma palavra com inicial maiúscula e até oito nomes candidatos; no máximo 256 tokens, sem cortar o histórico. Entradas temporárias ficam nesta sessão do Colab e não entram automaticamente no treino ou no Drive.


In [ ]:
MODO='escopo_preservado'  # ou 'eventos'; ambos experimentais
FALAS=[
    'O plano Azul custa 23 reais. O plano Verde custa 35 reais. Compare Azul e Verde.',
    'Corrigindo Azul: o custo real é 41 reais. Compare Azul e Verde.',
    'Imagine Azul por 19 reais, sem alterar os fatos. Compare Azul e Verde.',
    'A última hipótese aconteceu de verdade. Agora ela é fato. Compare Azul e Verde.',
    'Esqueça a simulação e use os preços reais corrigidos. Compare Azul e Verde.',
]
p=Path('/content/falas-eventos.json');p.write_text(json.dumps(FALAS,ensure_ascii=False))
RUNNER = """
import sys,json,torch
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from base import ROOT
from rede_eventos import carregar
from normalizacao import codificar
from modelo import lote,executar
from decodificar import proposta_limitada
from dados import EVENTOS
from tokenizers import Tokenizer
torch.set_num_threads(1)
m,_=carregar(Path(sys.argv[1])/'eventos/pesos.pt')
if sys.argv[4]=='escopo_preservado':
    sys.path.insert(0,sys.argv[3])
    from piloto import EscopoPreservado
    anterior,_=carregar()
    m=EscopoPreservado(m,anterior)
m.eval()
tok=Tokenizer.from_file(str(ROOT/'artefatos/linguagem_profunda/tokenizer.json'));tok.encode_special_tokens=True
falas=json.loads(Path(sys.argv[2]).read_text())
for n in range(1,len(falas)+1):
    e={'turnos':falas[:n],'argumentos':[None]*3,'referente':None}
    item=codificar(tok,e);x,l=lote([item],tok.token_to_id('<pad>'))
    with torch.no_grad():ls=m(x,l)
    p=proposta_limitada(item,ls,0)
    print(json.dumps({'turno':n,'evento_previsto':EVENTOS[int(ls['evento'][0].argmax())],
        'proposta':p,'execucao':executar(p)},ensure_ascii=False))
"""
subprocess.check_call([sys.executable,'-c',RUNNER,str(LAB),str(p),str(PILOTO),MODO])


## Reproduzir o treino, opcionalmente

Os pesos escolhidos já vêm na branch. Scripts `dados.py` e `treino.py` permitem uma nova execução em pastas novas; cada braço executa 600 atualizações e guarda Adam/RNG para retomada local. Repetir este pequeno corpus não equivale a ampliar raciocínio ou conversa livre. Esta rodada foi executada em CPU. O notebook foi validado localmente, mas não executado numa sessão real do Colab.
